# Airline Satisfaction — CatBoost baseline

This notebook trains a binary classifier for `satisfaction`, evaluates it with **ROC AUC** on a stratified holdout, then fits on all labeled rows and writes `submission_catboost.csv`. It assumes this notebook and the two CSV files are in the same folder.


In [2]:
# Run once if CatBoost is missing from this notebook environment.
%pip install catboost


Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix


In [11]:
# Files are next to this notebook. The training CSV was named "train (1).csv".
DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "train (1).csv"
TEST_PATH = DATA_DIR / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = "satisfaction"
ID_COL = "id"

print("train:", train.shape, "test:", test.shape)
display(train.head())
print("\nTarget distribution:")
print(train[TARGET].value_counts(dropna=False))
print("\nMissing values (training):")
print(train.isna().sum().sort_values(ascending=False).head(10))


train: (699635, 23) test: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False



Target distribution:
satisfaction
False    389296
True     310339
Name: count, dtype: int64

Missing values (training):
Arrival Delay in Minutes      292
id                              0
Leg room service                0
Class                           0
Type of Travel                  0
Customer Type                   0
Gender                          0
Departure Delay in Minutes      0
Cleanliness                     0
Checkin service                 0
dtype: int64


In [12]:
# Separate identifiers and target; do not train on the ID by default.
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COL]).copy()

# CatBoost accepts categorical columns directly; represent missing categories explicitly.
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in cat_features:
    X[col] = X[col].fillna("__MISSING__").astype(str)
    X_test[col] = X_test[col].fillna("__MISSING__").astype(str)

# Ensure numeric columns use consistent numeric values and preserve missing values for CatBoost.
for col in X.columns.difference(cat_features):
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

print("Categorical features:", cat_features)
print("Positive class rate:", y.mean())


Categorical features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Positive class rate: 0.44357272006117476


In [13]:
# Split labeled data into training and validation sets. This must run before model comparison.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Compare CatBoost tree depths on the same stratified validation split.
# Keep all other settings fixed so the comparison isolates the effect of depth.
depths_to_try = [7, 8, 9]
models_by_depth = {}
scores_by_depth = {}

for depth_value in depths_to_try:
    print(f'\nTraining CatBoost with depth={depth_value}')
    candidate = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.05,
        depth=depth_value,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=300,
        early_stopping_rounds=150,
        allow_writing_files=False,
        thread_count=-1,
    )
    candidate.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
    )
    candidate_proba = candidate.predict_proba(X_valid)[:, 1]
    candidate_auc = roc_auc_score(y_valid, candidate_proba)
    models_by_depth[depth_value] = candidate
    scores_by_depth[depth_value] = candidate_auc
    print(f'depth={depth_value}: validation ROC AUC={candidate_auc:.6f}; best iteration={candidate.best_iteration_}')

best_depth = max(scores_by_depth, key=scores_by_depth.get)
model = models_by_depth[best_depth]
valid_proba = model.predict_proba(X_valid)[:, 1]
print(f'\nSelected depth: {best_depth} (AUC={scores_by_depth[best_depth]:.6f})')





Training CatBoost with depth=7
0:	test: 0.9374761	best: 0.9374761 (0)	total: 439ms	remaining: 18m 17s
300:	test: 0.9565964	best: 0.9565964 (300)	total: 1m 37s	remaining: 11m 50s
600:	test: 0.9572682	best: 0.9572682 (600)	total: 3m 24s	remaining: 10m 46s
900:	test: 0.9575268	best: 0.9575268 (900)	total: 5m 37s	remaining: 9m 58s
1200:	test: 0.9576369	best: 0.9576395 (1196)	total: 8m 7s	remaining: 8m 47s
1500:	test: 0.9577046	best: 0.9577106 (1439)	total: 9m 41s	remaining: 6m 26s
1800:	test: 0.9577429	best: 0.9577554 (1753)	total: 11m 13s	remaining: 4m 21s
2100:	test: 0.9578005	best: 0.9578005 (2100)	total: 12m 48s	remaining: 2m 25s
2400:	test: 0.9578205	best: 0.9578205 (2400)	total: 14m 20s	remaining: 35.5s
2499:	test: 0.9578340	best: 0.9578340 (2499)	total: 14m 51s	remaining: 0us

bestTest = 0.9578340348
bestIteration = 2499

depth=7: validation ROC AUC=0.957834; best iteration=2499

Training CatBoost with depth=8
0:	test: 0.9438315	best: 0.9438315 (0)	total: 431ms	remaining: 17m 56s
3

## Optional threshold-based check

The competition is scored by ROC AUC, so the probability ranking above is the key result. The next cell is only an optional classification report at threshold 0.5; do not submit these hard labels.


In [14]:
valid_pred = (valid_proba >= 0.5).astype(int)
print(classification_report(y_valid, valid_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_valid, valid_pred))


              precision    recall  f1-score   support

           0     0.9244    0.9429    0.9335     77859
           1     0.9265    0.9032    0.9147     62068

    accuracy                         0.9253    139927
   macro avg     0.9254    0.9230    0.9241    139927
weighted avg     0.9253    0.9253    0.9252    139927

Confusion matrix:
 [[73410  4449]
 [ 6006 56062]]


In [15]:
# Refit using the full labeled dataset and the best depth/iteration count from validation.
best_iterations = model.best_iteration_ + 1 if model.best_iteration_ is not None else None
if best_iterations is None or best_iterations < 1:
    best_iterations = 1500

final_model = CatBoostClassifier(
    iterations=best_iterations,
    learning_rate=0.05,
    depth=best_depth,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1,
)
final_model.fit(X, y, cat_features=cat_features)
test_proba = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_proba})
submission_path = DATA_DIR / "submission_catboost.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path.resolve())
print(submission.head())
print("Rows:", len(submission), "IDs unique:", submission[ID_COL].is_unique)
print("Probabilities in [0, 1]:", submission[TARGET].between(0, 1).all())


0:	total: 262ms	remaining: 10m 55s
200:	total: 1m	remaining: 11m 35s
400:	total: 2m 13s	remaining: 11m 40s
600:	total: 3m 21s	remaining: 10m 35s
800:	total: 4m 14s	remaining: 8m 58s
1000:	total: 5m 23s	remaining: 8m 4s
1200:	total: 6m 36s	remaining: 7m 8s
1400:	total: 7m 43s	remaining: 6m 3s
1600:	total: 9m	remaining: 5m 3s
1800:	total: 10m 14s	remaining: 3m 58s
2000:	total: 11m 13s	remaining: 2m 47s
2200:	total: 12m 15s	remaining: 1m 39s
2400:	total: 13m 30s	remaining: 33.4s
2499:	total: 14m 7s	remaining: 0us
Saved: C:\Users\Asus\OneDrive\Desktop\Predicting_Airline_Satisfaction\submission_catboost.csv
       id  satisfaction
0  699635      0.234019
1  699636      0.895494
2  699637      0.902130
3  699638      0.074841
4  699639      0.065179
Rows: 299844 IDs unique: True
Probabilities in [0, 1]: True
